# Colour Brings People Together
### Part B · Auditing the UN SDG palette (45 min)

In Part A's card game you talked about the colours behind English words. Now you'll test a few of the UN's 17 SDG colours against thousands of speakers of seven languages.

<img src="https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/images/sdg_wheel.png" alt="The 17 Sustainable Development Goals" width="220">

The UN uses the same 17 colours in every country and language. Could the colours alone tell the goals apart, for every eye and in every language? You'll check two things, then see what an AI says; in Part C you'll fix the palette.

| | The question |
|---|---|
| **Perceptual** | Can people tell the colours apart with a colour vision deficiency, or an ageing eye? |
| **Cognitive** | Do speakers agree on each colour's name, and do the names in different languages mean the same colour? |

Run each cell with **Shift + Enter**. Change only the grey forms and the menus.

In [ ]:
#@title Run me first: load the masterclass toolkit
#@markdown Takes about 30 seconds the first time. You're ready when it says **Toolkit ready ✓**.
TOOLKIT_URL = "https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/colour_masterclass.py"
import os, importlib, urllib.request
try:                                   # always fetch the current version
    urllib.request.urlretrieve(TOOLKIT_URL, "colour_masterclass.py")
except Exception as e:
    if not os.path.exists("colour_masterclass.py"):
        raise RuntimeError("Could not download the toolkit. Check your internet connection, or upload "
                           "colour_masterclass.py in Colab's Files panel (folder icon, left).") from e
    print("Couldn't refresh the toolkit, so using the copy already here.")
import colour_masterclass
importlib.reload(colour_masterclass)
from colour_masterclass import *

## 1 · Meet the palette

The 17 SDG colours. Which ones look alike to you?

*They sit on mid-grey: differences are easiest to see near the background's lightness, so grey keeps both dark and light colours distinguishable.*

In [ ]:
#@title The 17 SDG colours
show_palette(SDG.hex.tolist(), labels=SDG.index.tolist(), captions=SDG.goal.tolist())

## 2 · How the palette looks to different eyes

About 1 in 12 men and 1 in 200 women of Northern European descent have a colour vision deficiency, most often confusing reds and greens; and with age, the eye's lens yellows. The grid shows the palette for three deficiencies and at ages 40, 60 and 80, simulated by the Colour Spinner. Look for colours that **merge** further down.

In [ ]:
#@title The palette under each viewing condition
prefetch_simulations(SDG.hex.tolist())      # ask the Colour Spinner for all 17 colours at once
show_conditions(SDG.hex.tolist(), labels=[str(i) for i in SDG.index])

## 3 · Your palette, your language and your audience

- **Test colours:** three to five SDG colours, as for a chart, a map or signs. Include a pair that merged above. The defaults are 3, 6, 10, 14 and 15.
- **Your language:** the one you'd design in. If yours isn't among the seven, choose one you know well, often English.
- **Your audience:** your readers' languages. Include one very different from yours, such as **OtjiHimba** (northern Namibia), with seven colour terms: *serandu* covers English red, pink and orange.

In [ ]:
#@title Your choices { run: "auto" }
Subset = "3, 6, 10, 14, 15"  #@param {type:"string"}
YourLanguage = "British English"  #@param ["American English", "British English", "French", "Greek", "OtjiHimba", "Serbian", "Spanish"]
#@markdown **Your audience's languages** (tick one or more; your own language is always included):
AmericanEnglish = False  #@param {type:"boolean"}
BritishEnglish = False  #@param {type:"boolean"}
French = False  #@param {type:"boolean"}
Greek = True  #@param {type:"boolean"}
OtjiHimba = True  #@param {type:"boolean"}
Serbian = True  #@param {type:"boolean"}
Spanish = True  #@param {type:"boolean"}

Backend = "Colour Spinner API"   # facilitators: switch to "Copied from the browser" only if the Spinner can't be reached

ticked = {"American English": AmericanEnglish, "British English": BritishEnglish,
          "French": French, "Greek": Greek, "OtjiHimba": OtjiHimba, "Serbian": Serbian, "Spanish": Spanish}
AUDIENCE = [language for language, on in ticked.items() if on and language != YourLanguage]
LANGUAGES = [YourLanguage] + AUDIENCE                 # you first, then your audience
PICK = parse_subset(Subset)
show_subset(PICK, LANGUAGES)
check_choices(Subset, PICK, AUDIENCE)        # says what was understood, and any problems
show(f"<b>You:</b> {YourLanguage} &nbsp;·&nbsp; <b>Your audience:</b> {', '.join(AUDIENCE) or 'none ticked yet'}")

## 4 · Can everyone tell your colours apart?

For each test colour: its colour difference, ΔE₀₀, from the nearest other test colour, for each kind of vision. About 1 is just noticeable side by side; to be told apart **at a glance**, as in a legend or on a sign, colours need **10 or more**. The table lists the pairs below 10, and why.

<details><summary><b>What is ΔE₀₀?</b></summary>

CIEDE2000, the standard measure of how different two colours look, built from judgements by people with typical colour vision. For simulated colours it's an approximation.
</details>
<details><summary><b>Why 10?</b></summary>

A teaching rule of thumb: about 1 is the smallest difference you can see side by side, about 5 is clearly different side by side, and about 10 can be told apart at a glance without comparing.
</details>

In [ ]:
#@title Distance to the nearest test colour
ConfusableBelow = PART_B_THRESHOLD            # ΔE₀₀ 10, the threshold used throughout Part B

palette_strip(PICK)
nn_de, nn_id = nearest_neighbours(PICK)      # ΔE₀₀ to the nearest other test colour, per condition
plot_nearest_neighbours(nn_de, nn_id, ConfusableBelow)
show_at_risk(nn_de, nn_id, ConfusableBelow)

## 5 · Do you and your audience call these colours the same?

The Colour Spinner shows what speakers call a colour, from about 81,000 names given by 2,450 speakers.

**Try it while the next cell runs:** click the Spinner's logo, choose your language in its **top** menu (it opens in American English), and type a test colour's hex code into the **Picker**. The **Namer** gives the top name and its confidence; the **Mapper**'s top-left tile is that name's best example. Then switch to an audience language.

The cell does this for every test colour and language:
- **Nameability:** the Namer's confidence. Below 50%, speakers disagree: the colour sits between names.
- **Translatability:** do the names in different languages point to the same colour, i.e. are their best examples close?

<details><summary><b>Why doesn't the Spinner's name match what I'd say?</b></summary>

It gives the name most speakers of that language chose, not yours: individuals, regions and professions differ. That's exactly why a design shouldn't rely on one person's naming, including the designer's.
</details>
<details><summary><b>Why only seven languages, and so few names?</b></summary>

The Spinner covers the indispensable vocabulary of seven languages it has data for: from 7 terms in OtjiHimba to 47 in American English (32 in British English, Greek, Serbian and Spanish; 27 in French). Treat it as a baseline, not the whole language.
</details>

In [ ]:
#@title Ask the Colour Spinner for every name
spinner_link()                             # the logo, as a link that opens directly
boxes_first(Backend, PICK, LANGUAGES)      # only with "Copied from the browser": shows boxes to fill in first
show(small("Asking the Colour Spinner for every test colour in every language; this takes a minute, please wait…"))
spinner = connect(Backend)

rows, problems = [], []
for sdg in PICK:
    for language in LANGUAGES:
        colour = SDG.hex[sdg]
        try:
            namer = spinner.name(colour, language)                  # Namer: top name and its confidence
        except Exception as e:
            problems.append(f"SDG {sdg} in {language}: {e}"); continue
        best = safe(spinner.best_example, namer["name"], language)  # Mapper: where that name is centred
        rows.append({"sdg": sdg, "hex": colour, "language": language,
                     "role": "your language" if language == YourLanguage else "audience",
                     "name": namer["name"], "romanized": namer.get("romanized"),   # Latin spelling, e.g. kokkino
                     "confidence": namer["confidence"], "best_example": best,
                     "drift": de00(colour, best) if best else np.nan})  # how far the name pulls the colour
naming = pd.DataFrame(rows)

report_problems(problems)
show_names_table(naming, LANGUAGES, you=YourLanguage)

### Do your names point where your audience's do?

In [ ]:
#@title Translatability: run to see the best examples of the predicted names
show_translatability(naming, PICK, LANGUAGES, ConfusableBelow, source=YourLanguage)

Each swatch is the best example of a language's name for the test colour; under it, its ΔE₀₀ from **your** language's best example. Below 10, the names translate well; 10 or more, a translation lands on a different colour.

**Where does each language's name pull the colour?** Each panel: a test colour (numbered circle) on the CIELAB a*–b* plane, seen from above, with lines to each language's best example. The longer the line, the further that name pulls the colour. All panels share one scale.

In [ ]:
#@title Where each name pulls the colour
plot_name_map(naming, PICK, LANGUAGES)

## 6 · The colour identity scorecard (optional, if time allows)

Each point is a pair of your test colours. **Up:** how different they look (ΔE₀₀). **Across:** how differently their names are used, from the Spinner's Thesaurus data: 0 is the same name, 1 unrelated names. Green corner: distinct both ways; pink corner: alike both ways. Try other **Language** and **Vision** choices.

In [ ]:
#@title Scorecard
#@markdown Choose whose names to compare, and which kind of vision the ΔE₀₀ is for.
live_controls({"language": ("Language", ["My language"] + AUDIENCE, "My language"),
               "vision":   ("Vision", VISIONS, "Typical")},
              lambda language, vision: plot_pair_scorecard(scorecard_pairs(PICK, naming, {"My language": YourLanguage}.get(language, language), vision)))

## 7 · Does an AI name them the way your audience does?

Ask the AI you use day to day to name your test colours in all your languages, then compare. Look for **confident names where the AI differs**, and its name means a different colour to speakers: bring those to the group.

<details><summary><b>Which AI should I use?</b></summary>

The one you'd really use for design work: ChatGPT, Claude, Gemini, Copilot or another. Use the same one in Part C.
</details>
<details><summary><b>Why would the AI disagree with speakers?</b></summary>

AI models learn mostly from English text, and tend to map names onto the same 'universal' prototypes in every language, missing how each language divides colour.
</details>

In [ ]:
#@title The prompt for your AI
lines = "\n".join(numbered_items(naming, "name"))          # one number per test colour
himba = "\n(OtjiHimba is spoken by the Himba of northern Namibia.)" if "OtjiHimba" in LANGUAGES else ""
reply_format = " | ".join(f"{language}: name" for language in LANGUAGES)
PROMPT_B = f"""You are helping with a cross-cultural colour naming study.
For each numbered colour code below, give the single colour name that most native speakers would use for it in each of these languages: {", ".join(LANGUAGES)}. Write each name in that language and in its usual script; for a non-Latin script, add the Latin spelling in brackets.{himba}
If you're not certain of a name, give your best guess rather than leaving it out: we will compare the answers with empirical data.

{lines}

Reply with one line per number, keeping the numbers above ({", ".join(map(str, PICK))}), in this format and nothing else:
{PICK[0]}. {reply_format}
…"""
show_prompt(PROMPT_B, PASTE, rows=naming[["sdg", "hex", "language"]], kind="name")

In [ ]:
#@title Compare the AI's names with your audience's
ai_names = collected_answers()                         # the names read from the pasted reply
ai_names["language"] = ai_names["language"].map(canonical_language)

comparison = naming.merge(ai_names, on=["hex", "language"], how="left")
comparison["agrees"] = [same_name(ai, name, rom)    # any script; ignores case and accents
                        for ai, name, rom in zip(comparison.ai_name, comparison.name, comparison.romanized)]
comparison = where_ai_names_point(comparison, spinner)   # Mapper: where each differing AI name is centred
show_name_comparison(comparison, ConfusableBelow, you=YourLanguage)
plot_agreement_by_language(comparison, LANGUAGES)

## 8 · Save your results for Part C

Run the next cell to download `partB_results.csv`, then click **Open Part C** underneath it. You'll upload the file at the start of Part C, to carry on with the same names.

In [ ]:
#@title Download your results
out = naming.merge(ai_names, on=["hex", "language"], how="left") if "ai_names" in globals() else naming
out.to_csv("partB_results.csv", index=False)
download("partB_results.csv")
open_part_c()                                # a button to Part C

## Before you share with the room

If you're working in a pair, agree with your partner on one sentence for each; if you're working on your own, note down one sentence for each:

1. Which test colour works best for **all** your audience, and which worst?
2. Is your worst colour failing **perceptually** (hard to tell apart), **linguistically** (your audience calls it something else), or **both**?
3. Where did the AI's names differ from your audience's? Does it look like a default to English or to "universal" colour categories?

In Part C you'll fix the palette for each audience.

---
#### Credits and references

**SDG colours and wheel:** United Nations, [Sustainable Development Goals communications materials](https://www.un.org/sustainabledevelopment/news/communications-material/); [un.org/sustainabledevelopment](https://www.un.org/sustainabledevelopment). *The content of this publication has not been approved by the United Nations and does not reflect the views of the United Nations or its officials or Member States.*

**Colour naming data and simulations:** the Colour Spinner, [colornaming.net](https://colornaming.net); Sinkeviciute, Ahmed, Stutters, Koliousis & Mylonas (2026), under review. Colour vision deficiencies: Viénot, Brettel & Mollon (1999); ageing: Pokorny, Smith & Lutze (1987).

**Colour science:** CIELAB and CIEDE2000 with the [colour-science](https://www.colour-science.org) library.